# Proyecto ING200 — Optimización 2026-2
## Fiesta de Verano de Villa Peumayén

**Integrantes:** _(completar)_

Este notebook resuelve las preguntas (a), (b), (c) y (d) del proyecto.

**Instrucciones de ejecución:** ejecutar las celdas en orden. La primera celda instala `gurobipy`
y monta la licencia de Gurobi desde Google Drive; si la licencia está en otra ruta, ajustar
`GRB_LICENSE_FILE`.

In [ ]:
# =====================================================================
# ENTORNO DE EJECUCION (Google Colab + licencia de Gurobi)
# =====================================================================
!pip install -q gurobipy

import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
os.environ['GRB_LICENSE_FILE'] = '/content/drive/MyDrive/gurobi.lic'

import gurobipy as gp
gp.Model()   # verifica que la licencia esté disponible

## Datos del problema

Se cargan los datos y las funciones entregadas por el curso (`demanda`, `mostrar_parrilla`).

In [ ]:
# =============================================================================
# DATOS DEL PROYECTO 2026-2: FIESTA DE VERANO DE VILLA PEUMAYÉN
# ING200: Optimización
#
# Este archivo contiene los datos del proyecto y dos funciones de apoyo:
#   - demanda(artistas): estima el público que convoca un conjunto de artistas.
#   - mostrar_parrilla(parrilla): muestra una parrilla y su público por noche.
# Puede copiar todo su contenido directamente en su script.
#
# Los montos están en millones de pesos (MM$).
# =============================================================================

# -----------------------------------------------------------------------------
# NOCHES DEL FESTIVAL
# FACTOR_NOCHE[n]: fracción de la demanda que asiste la noche n.
# -----------------------------------------------------------------------------
NOCHES = ['Viernes', 'Sábado', 'Domingo']
FACTOR_NOCHE = {'Viernes': 0.90, 'Sábado': 1.00, 'Domingo': 0.85}

# -----------------------------------------------------------------------------
# RECINTOS
# recinto: capacidad (personas) y costo de arriendo por noche (MM$).
# Todos los recintos están disponibles las tres noches.
# -----------------------------------------------------------------------------
RECINTOS = {
    'Plaza de Armas':     {'capacidad': 12000, 'costo': 30},
    'Medialuna':          {'capacidad': 7500,  'costo': 18},
    'Gimnasio Municipal': {'capacidad': 4200,  'costo': 8},
}

# -----------------------------------------------------------------------------
# ARTISTAS
# nombre: género, nivel ('G' grande, 'M' mediano, 'P' pequeño),
#         popularidad (índice individual), costo (MM$) y noches disponibles.
# -----------------------------------------------------------------------------
ARTISTAS = {
    'La Sonora Copihue':           {'genero': 'Cumbia',  'nivel': 'G', 'popularidad': 1.08, 'costo': 52, 'noches': ['Viernes', 'Sábado', 'Domingo']},
    'Los Volcanes':                {'genero': 'Rock',    'nivel': 'G', 'popularidad': 0.97, 'costo': 40, 'noches': ['Viernes', 'Sábado', 'Domingo']},
    'Camila Arrebol':              {'genero': 'Pop',     'nivel': 'G', 'popularidad': 1.03, 'costo': 48, 'noches': ['Viernes', 'Sábado', 'Domingo']},
    'Kevo Flow':                   {'genero': 'Urbano',  'nivel': 'G', 'popularidad': 1.12, 'costo': 58, 'noches': ['Viernes', 'Sábado']},
    'Los Cumbiancheros del Ñuble': {'genero': 'Cumbia',  'nivel': 'M', 'popularidad': 0.94, 'costo': 18, 'noches': ['Viernes', 'Sábado', 'Domingo']},
    'Grupo Tierra Sur':            {'genero': 'Folclor', 'nivel': 'M', 'popularidad': 1.06, 'costo': 15, 'noches': ['Viernes', 'Sábado', 'Domingo']},
    'La Nena Real':                {'genero': 'Urbano',  'nivel': 'M', 'popularidad': 1.01, 'costo': 22, 'noches': ['Viernes', 'Sábado', 'Domingo']},
    'Las Hermanas Quintral':       {'genero': 'Pop',     'nivel': 'M', 'popularidad': 0.92, 'costo': 17, 'noches': ['Viernes', 'Sábado', 'Domingo']},
    'Ruta Cinco':                  {'genero': 'Rock',    'nivel': 'M', 'popularidad': 1.04, 'costo': 16, 'noches': ['Viernes', 'Sábado', 'Domingo']},
    'Banda Fogata':                {'genero': 'Cumbia',  'nivel': 'P', 'popularidad': 1.11, 'costo': 7,  'noches': ['Viernes', 'Sábado', 'Domingo']},
    'Conjunto Raíces del Valle':   {'genero': 'Folclor', 'nivel': 'P', 'popularidad': 0.98, 'costo': 5,  'noches': ['Viernes', 'Sábado', 'Domingo']},
    'Las Cantoras de Peumayén':    {'genero': 'Folclor', 'nivel': 'P', 'popularidad': 0.89, 'costo': 4,  'noches': ['Viernes', 'Sábado', 'Domingo']},
    'Niebla Austral':              {'genero': 'Rock',    'nivel': 'P', 'popularidad': 0.95, 'costo': 6,  'noches': ['Viernes', 'Sábado', 'Domingo']},
    'Tomás del Lago':              {'genero': 'Pop',     'nivel': 'P', 'popularidad': 1.09, 'costo': 8,  'noches': ['Viernes', 'Sábado', 'Domingo']},
    'Yisus Beat':                  {'genero': 'Urbano',  'nivel': 'P', 'popularidad': 0.91, 'costo': 6,  'noches': ['Viernes', 'Sábado', 'Domingo']},
}

# -----------------------------------------------------------------------------
# REGLAS ENTRE ARTISTAS
# TELONEROS: si se contrata al primero, el segundo debe presentarse la misma noche.
# EXCLUSIONES: no pueden ser contratados ambos.
# -----------------------------------------------------------------------------
TELONEROS = [('La Sonora Copihue', 'Banda Fogata')]
EXCLUSIONES = [('La Nena Real', 'Yisus Beat')]

# -----------------------------------------------------------------------------
# PARÁMETROS GENERALES
# -----------------------------------------------------------------------------
PRESUPUESTO = 250              # MM$: artistas + arriendo de recintos + plan de seguridad
UMBRAL_SEGURIDAD = 20000       # personas: si el público total del festival lo supera, se exige el plan
COSTO_SEGURIDAD = 25           # MM$

# -----------------------------------------------------------------------------
# PARÁMETROS DE LA FUNCIÓN DE DEMANDA (usados por la función demanda)
# -----------------------------------------------------------------------------
BASE_NIVEL = {'G': 4200, 'M': 1900, 'P': 750}
FACTOR_GENERO = {'Cumbia': 1.20, 'Urbano': 1.10, 'Pop': 1.00, 'Rock': 0.95, 'Folclor': 0.65}
ARRASTRE = 0.20                # aumento de la base de artistas M y P si en la noche hay un artista G
MISMO_GENERO = -0.15           # por cada par de artistas del mismo género
DOS_GRANDES = -0.10            # por cada par de artistas grandes
AFINIDAD = {('Cumbia', 'Urbano'): 0.06, ('Pop', 'Rock'): 0.05, ('Pop', 'Urbano'): 0.03}


def demanda(artistas):
    """Demanda esperada (personas) de un conjunto de artistas que se presentan la misma noche.

    No considera la noche ni el recinto: el público esperado de la noche n en el recinto r es
        min(FACTOR_NOCHE[n] * demanda(artistas), capacidad del recinto r),
    redondeado al entero más cercano.

    Ejemplo: demanda(['Kevo Flow', 'Ruta Cinco', 'Banda Fogata'])
    También acepta conjuntos de uno o dos artistas (útil, por ejemplo, para analizar cancelaciones).
    """
    artistas = list(artistas)
    hay_grande = any(ARTISTAS[a]['nivel'] == 'G' for a in artistas)
    base = 0.0
    for a in artistas:
        datos = ARTISTAS[a]
        b = BASE_NIVEL[datos['nivel']] * FACTOR_GENERO[datos['genero']] * datos['popularidad']
        if hay_grande and datos['nivel'] != 'G':
            b *= 1 + ARRASTRE
        base += b
    interaccion = 0.0
    for i in range(len(artistas)):
        for j in range(i + 1, len(artistas)):
            a1, a2 = ARTISTAS[artistas[i]], ARTISTAS[artistas[j]]
            if a1['genero'] == a2['genero']:
                interaccion += MISMO_GENERO
            interaccion += AFINIDAD.get((a1['genero'], a2['genero']), AFINIDAD.get((a2['genero'], a1['genero']), 0))
            if a1['nivel'] == 'G' and a2['nivel'] == 'G':
                interaccion += DOS_GRANDES
    return base * (1 + interaccion)


def mostrar_parrilla(parrilla, titulo='Parrilla del festival'):
    """Muestra una parrilla con su público y su costo por noche.

    parrilla: lista de tuplas (noche, recinto, [artista1, artista2, artista3]), por ejemplo
        [('Viernes', 'Medialuna', ['Kevo Flow', 'Ruta Cinco', 'Banda Fogata']), ...]
    El costo mostrado incluye artistas y recinto, pero no el plan de seguridad.
    """
    import matplotlib.pyplot as plt
    orden = {n: i for i, n in enumerate(NOCHES)}
    parrilla = sorted(parrilla, key=lambda t: orden[t[0]])
    print(f'{titulo}\n' + '-' * 100)
    total_pub, total_costo, publicos = 0, 0, []
    for noche, recinto, artistas in parrilla:
        pub = round(min(FACTOR_NOCHE[noche] * demanda(artistas), RECINTOS[recinto]['capacidad']))
        costo = sum(ARTISTAS[a]['costo'] for a in artistas) + RECINTOS[recinto]['costo']
        total_pub += pub; total_costo += costo; publicos.append(pub)
        nombres = ', '.join(f"{a} ({ARTISTAS[a]['nivel']})" for a in artistas)
        print(f'{noche:8s} | {recinto:18s} | {pub:6,d} personas | {costo:4d} MM$ | {nombres}')
    print('-' * 100 + f'\nTotal: {total_pub:,} personas | {total_costo} MM$ (sin plan de seguridad)')
    plt.figure(figsize=(7, 3.5))
    barras = plt.bar([p[0] for p in parrilla], publicos, color=['#1565c0', '#c62828', '#2e7d32'][:len(parrilla)])
    for b, (noche, recinto, _) in zip(barras, parrilla):
        plt.text(b.get_x() + b.get_width() / 2, b.get_height(), recinto, ha='center', va='bottom', fontsize=9)
    plt.ylabel('Público esperado'); plt.title(titulo); plt.tight_layout(); plt.show()

---
# Pregunta 1(a) — Formulación del modelo

## Idea central

La demanda $D(c)$ **no es aditiva**: depende de la interacción entre pares de artistas y del efecto
arrastre de los artistas grandes. Sin embargo, el enunciado fija que **cada noche se presentan exactamente
tres artistas**, de modo que el conjunto de programas posibles para una noche es finito y pequeño:
todos los tríos.

Por lo tanto la demanda de un trío puede **calcularse a priori como un dato**:

$$D(c)\ \text{conocido}\quad\Longrightarrow\quad \text{Pub}(c,n,r)=\operatorname{round}\big(\min\{\phi_n D(c),K_r\}\big)\ \text{es un parámetro}$$

Toda la no linealidad (producto de bases, ajuste de interacción, arrastre y mínimo con la capacidad)
se resuelve **al construir los parámetros**. El modelo que ve el solver es lineal entero puro: **elige
paquetes (trío, noche, recinto) ya valorizados**.

## Conjuntos

| Conjunto | Definición |
| :--- | :--- |
| $\mathcal{A}$ | Artistas candidatos ($\|\mathcal{A}\|=15$) |
| $\mathcal{N}$ | Noches ($\|\mathcal{N}\|=3$) |
| $\mathcal{R}$ | Recintos ($\|\mathcal{R}\|=3$) |
| $\mathcal{C}$ | Tríos que respetan la regla del telonero |
| $\Omega$ | Asignaciones factibles $(c,n,r)\in\mathcal{C}\times\mathcal{N}\times\mathcal{R}$ con todos los artistas de $c$ disponibles la noche $n$ |

## Parámetros

| Parámetro | Significado |
| :--- | :--- |
| $\text{Pub}_{cnr}$ | $\operatorname{round}(\min\{\phi_nD(c),K_r\})$ |
| $\text{Cost}_{cnr}$ | $\sum_{a\in c} w_a + c_r$ |
| $\text{Disp}_{cn}$ | $1$ si todos los artistas de $c$ están disponibles la noche $n$ |
| $U=20.000$ | Umbral del plan de seguridad |
| $S=25$ | Costo del plan de seguridad (MM$) |
| $B=250$ | Presupuesto total (MM$) |
| $M=16.000$ | $=3\cdot 12.000-U$: cota exacta del exceso de público sobre el umbral |

## Variables

$$X_{cnr}\in\{0,1\}:\ 1\text{ si el trío } c\text{ toca la noche } n\text{ en el recinto } r$$
$$Y\in\{0,1\}:\ 1\text{ si se implementa el plan de seguridad}$$

## Objetivo

$$\max\ Z=\sum_{(c,n,r)\in\Omega}\text{Pub}_{cnr}\,X_{cnr}$$

## Restricciones

**(R1) Exactamente un trío y un recinto por noche**
$$\sum_{(c,n,r)\in\Omega}X_{cnr}=1\qquad\forall n\in\mathcal{N}$$
*Una sola parrilla por noche. Como $X$ indexa tríos (de 3 artistas), esto codifica a la vez
"exactamente tres artistas" y "todos en el mismo recinto".*

**(R2) Cada artista a lo más una vez**
$$\sum_{(c,n,r):\,a\in c}X_{cnr}\le 1\qquad\forall a\in\mathcal{A}$$

**(R3) Disponibilidad por noche**
$$X_{cnr}\le \text{Disp}_{cn}\qquad\forall(c,n,r)\in\Omega$$
*Implementada filtrando el universo de variables a $\Omega$ (equivalente exacto).*

**(R4) Exclusión La Nena Real – Yisus Beat (regla de festival)**
$$\sum_{(c,n,r):\,\text{Nena}\in c}X_{cnr}+\sum_{(c,n,r):\,\text{Yisus}\in c}X_{cnr}\le 1$$
*La suma recorre a los dos artistas por separado: si un trío contuviera a ambos aportaría 2 y violaría
la restricción. Sumar cada trío una sola vez sería un error (admitiría el trío con ambos).*

**(R5) Plan de seguridad**
$$\sum_{(c,n,r)}\text{Pub}_{cnr}X_{cnr}-U\le M\,Y$$

**(R6) Presupuesto**
$$\sum_{(c,n,r)}\text{Cost}_{cnr}X_{cnr}+S\,Y\le B$$

## ¿Por qué es lineal?

1. La no linealidad de $D(c)$ es **de conjunto, no de decisión**: el trío es la unidad de decisión, y su
   demanda se evalúa una vez por trío.
2. El $\min$ con la capacidad y el arrastre quedan **dentro del dato** $\text{Pub}_{cnr}$.
3. Lo que resta es un modelo de **partición de conjuntos**: objetivo y restricciones lineales.

*Nota de generalidad: si creciera el número de artistas por noche esto dejaría de ser tratable y habría
que usar variables por artista $y_{an}$ y de par $w_{aa'n}=y_{an}y_{a'n}$ linealizadas. Aquí la
enumeración es exacta y preferible.*

In [ ]:
# =====================================================================
# (a) Universo de variables y parámetros
# =====================================================================
import itertools

# --- Conjunto C: tríos que respetan la regla del telonero -------------
C = []
for c in itertools.combinations(ARTISTAS.keys(), 3):
    if 'La Sonora Copihue' in c and 'Banda Fogata' not in c:
        continue          # si se contrata a La Sonora, Banda Fogata debe tocar la misma noche
    C.append(c)

def disponible(c, n):
    """True si todos los artistas del trío están disponibles la noche n."""
    return all(n in ARTISTAS[a]['noches'] for a in c)

# --- Omega: (trío, noche, recinto) factibles  (restricción R3) --------
OMEGA = [(c, n, r) for c in C for n in NOCHES for r in RECINTOS if disponible(c, n)]

# --- Parámetros precalculados -----------------------------------------
PUB = {(c, n, r): round(min(FACTOR_NOCHE[n] * demanda(c), RECINTOS[r]['capacidad']))
       for c, n, r in OMEGA}                                    # público esperado
COST = {(c, n, r): sum(ARTISTAS[a]['costo'] for a in c) + RECINTOS[r]['costo']
        for c, n, r in OMEGA}                                   # costo directo
PUB_SIN = {(c, n, r, a): round(min(FACTOR_NOCHE[n] * demanda([x for x in c if x != a]),
                                   RECINTOS[r]['capacidad']))
           for c, n, r in OMEGA for a in c}                     # público si cancela a

M_SEG = 3 * 12000 - UMBRAL_SEGURIDAD     # 16.000: cota exacta del exceso sobre el umbral
M_ROB = 12000                            # cota del público de una noche (capacidad máxima)

print(f'|C| = {len(C)} tríos válidos')
print(f'|Omega| = {len(OMEGA)} asignaciones (trío, noche, recinto)')

# --- Verificación con el ejemplo del enunciado ------------------------
d = demanda(['La Sonora Copihue', 'Kevo Flow', 'Yisus Beat'])
print(f'\nEjemplo del enunciado: D = {d:,.1f} (esperado 10.021,1)')
print(f'Viernes en Plaza de Armas = {round(min(FACTOR_NOCHE["Viernes"] * d, 12000)):,} (esperado 9.019)')

---
# Pregunta 1(b) — Implementación y solución

- `MIPGap = 0` para optimalidad exacta.
- **Desempate por menor costo total** en dos etapas: (1) maximizar público; (2) fijar el óptimo y
  minimizar el costo total (incluido el plan de seguridad).

In [ ]:
# =====================================================================
# (b) Modelo de optimización
# =====================================================================
def construir_modelo(presupuesto=PRESUPUESTO, alpha=None, minimo_noche=None,
                     max_pequenos=None, min_grandes=None, nombre='Festival', verbose=False):
    """Construye el modelo de (a) y devuelve (modelo, X, Y, publico, costo, opciones).

    alpha         : (c) II  robustez, Pub(c\\{a}) >= alpha * Pub(c)
    minimo_noche  : (d) III piso de público por noche, dict noche -> personas
    max_pequenos  : (c) II  máximo de artistas pequeños por noche
    min_grandes   : (d) III mínimo de artistas grandes por noche
    """
    opciones = OMEGA
    if max_pequenos is not None:
        opciones = [k for k in OMEGA
                    if sum(1 for a in k[0] if ARTISTAS[a]['nivel'] == 'P') <= max_pequenos]

    m = gp.Model(nombre)
    m.Params.MIPGap = 0
    m.Params.MIPGapAbs = 0
    m.Params.OutputFlag = 1 if verbose else 0

    X = m.addVars(opciones, vtype=GRB.BINARY, name='X')
    Y = m.addVar(vtype=GRB.BINARY, name='Y')

    publico = gp.quicksum(PUB[k] * X[k] for k in opciones)
    costo = gp.quicksum(COST[k] * X[k] for k in opciones)

    # (R1) exactamente un trío y un recinto por noche
    m.addConstrs((gp.quicksum(X[k] for k in opciones if k[1] == n) == 1 for n in NOCHES),
                 name='R1_un_trio_por_noche')

    # (R2) cada artista a lo más una vez en el festival
    for i, a in enumerate(ARTISTAS):
        m.addConstr(gp.quicksum(X[k] for k in opciones if a in k[0]) <= 1,
                    name=f'R2_artista_{i}')

    # (R3) disponibilidad: ya incorporada en Omega

    # (R4) exclusión a nivel de festival (cuenta a cada artista por separado)
    m.addConstr(gp.quicksum(X[k] for k in opciones if 'La Nena Real' in k[0]) +
                gp.quicksum(X[k] for k in opciones if 'Yisus Beat' in k[0]) <= 1,
                name='R4_exclusion')

    # (R5) plan de seguridad
    m.addConstr(publico - UMBRAL_SEGURIDAD <= M_SEG * Y, name='R5_seguridad')

    # (R6) presupuesto
    m.addConstr(costo + COSTO_SEGURIDAD * Y <= presupuesto, name='R6_presupuesto')

    # ---- condiciones adicionales de (c) II y (d) III ----
    if alpha is not None:
        nrob = 0
        for k in opciones:
            c, n, r = k
            for a in c:
                m.addConstr(PUB_SIN[(c, n, r, a)] >= alpha * PUB[k] - M_ROB * (1 - X[k]),
                            name=f'rob_{nrob}')
                nrob += 1
    if minimo_noche:
        for n, piso in minimo_noche.items():
            m.addConstr(gp.quicksum(PUB[k] * X[k] for k in opciones if k[1] == n) >= piso,
                        name=f'piso_{n}')
    if min_grandes is not None:
        for n in NOCHES:
            m.addConstr(gp.quicksum(X[k] for k in opciones
                                    if k[1] == n
                                    and any(ARTISTAS[a]['nivel'] == 'G' for a in k[0])) >= min_grandes,
                        name=f'grandes_{n}')

    m.setObjective(publico, GRB.MAXIMIZE)
    return m, X, Y, publico, costo, opciones


def armar_solucion(elegidos, seguridad, presupuesto):
    """Empaqueta la solución en un diccionario legible."""
    orden = {n: i for i, n in enumerate(NOCHES)}
    elegidos = sorted(elegidos, key=lambda k: orden[k[1]])
    parrilla = [(n, r, list(c)) for c, n, r in elegidos]
    pub_noche = {n: round(min(FACTOR_NOCHE[n] * demanda(c), RECINTOS[r]['capacidad']))
                 for n, r, c in parrilla}
    costo_art = sum(ARTISTAS[a]['costo'] for _, _, c in parrilla for a in c)
    costo_rec = sum(RECINTOS[r]['costo'] for _, r, _ in parrilla)
    return {'parrilla': parrilla,
            'pub_noche': pub_noche,
            'pub_total': sum(pub_noche.values()),
            'costo_artistas': costo_art,
            'costo_recintos': costo_rec,
            'costo_seguridad': COSTO_SEGURIDAD if seguridad else 0,
            'costo_total': costo_art + costo_rec + (COSTO_SEGURIDAD if seguridad else 0),
            'seguridad': seguridad,
            'presupuesto': presupuesto}


def resolver(presupuesto=PRESUPUESTO, **kwargs):
    """Dos etapas: (1) máximo público  (2) fijado el óptimo, mínimo costo total."""
    m, X, Y, publico, costo, opciones = construir_modelo(presupuesto, **kwargs)
    m.optimize()
    if m.Status != GRB.OPTIMAL:
        return None
    Z = round(m.ObjVal)

    m2, X2, Y2, publico2, costo2, op2 = construir_modelo(presupuesto, nombre='Desempate', **kwargs)
    m2.addConstr(publico2 == Z, name='fijar_publico_optimo')
    m2.setObjective(costo2 + COSTO_SEGURIDAD * Y2, GRB.MINIMIZE)
    m2.optimize()
    assert m2.Status == GRB.OPTIMAL, 'desempate no óptimo'

    elegidos = [k for k in op2 if X2[k].X > 0.5]
    return armar_solucion(elegidos, Y2.X > 0.5, presupuesto)


def validar(sol):
    """Verificación independiente de todas las condiciones del enunciado."""
    err, parrilla = [], sol['parrilla']
    if sorted(n for n, _, _ in parrilla) != sorted(NOCHES):
        err.append('no hay exactamente una parrilla por noche')
    todos = [a for _, _, c in parrilla for a in c]
    for n, r, c in parrilla:
        if len(c) != 3:
            err.append(f'{n}: no son 3 artistas')
        if not disponible(c, n):
            err.append(f'{n}: artista no disponible')
    dup = [a for a in set(todos) if todos.count(a) > 1]
    if dup:
        err.append(f'artistas repetidos: {dup}')
    for a, b in TELONEROS:
        if a in todos:
            noche = [n for n, _, c in parrilla if a in c][0]
            if b not in [x for n, _, c in parrilla if n == noche for x in c]:
                err.append(f'telonero: {b} no acompaña a {a}')
    for a, b in EXCLUSIONES:
        if a in todos and b in todos:
            err.append(f'exclusión violada: {a} y {b}')
    if sol['pub_total'] > UMBRAL_SEGURIDAD and not sol['seguridad']:
        err.append('falta plan de seguridad')
    if sol['costo_total'] > sol['presupuesto']:
        err.append('presupuesto excedido')
    return err


def mostrar(sol, titulo=''):
    """Reporte completo de una solución."""
    print(f'\n=== {titulo} (presupuesto {sol["presupuesto"]} MM$) ===')
    for n, r, c in sol['parrilla']:
        detalle = ', '.join(f"{a} ({ARTISTAS[a]['nivel']})" for a in c)
        print(f'{n:8s} | {r:18s} | {sol["pub_noche"][n]:6,d} personas | {detalle}')
    print(f'{"":8s} | {"TOTAL":18s} | {sol["pub_total"]:6,d} personas')
    print(f'Costo: artistas {sol["costo_artistas"]} + recintos {sol["costo_recintos"]} '
          f'+ seguridad {sol["costo_seguridad"]} = {sol["costo_total"]} MM$')
    print(f'Plan de seguridad: {"SÍ" if sol["seguridad"] else "NO"}')
    print('Validación:', validar(sol) or 'todas las condiciones se cumplen')


# ------------------------- SOLUCIÓN BASE ------------------------------
base = resolver(PRESUPUESTO)
mostrar(base, 'SOLUCIÓN ÓPTIMA BASE')
mostrar_parrilla(base['parrilla'], titulo='Parrilla óptima base')

---
# Pregunta 1(c) I — Dependencia de un solo artista

Para cada noche de la solución base se recalcula el público con los dos artistas restantes, en el mismo
recinto y con el mismo factor de noche.

In [ ]:
# =====================================================================
# (c) I  Público si cancela cada artista de la noche
# =====================================================================
def analisis_cancelaciones(sol):
    filas = []
    for n, r, c in sol['parrilla']:
        pubs = {a: round(min(FACTOR_NOCHE[n] * demanda([x for x in c if x != a]),
                             RECINTOS[r]['capacidad'])) for a in c}
        filas.append({'noche': n, 'recinto': r, 'trio': list(c), 'publico': sol['pub_noche'][n],
                      'sin': pubs, 'peor': min(pubs.values()),
                      'ratio': min(pubs.values()) / sol['pub_noche'][n],
                      'peor_artista': min(pubs, key=pubs.get)})
    return filas


cancel = analisis_cancelaciones(base)
for f in cancel:
    print(f"\n{f['noche']} — {f['recinto']} — público base {f['publico']:,}")
    for a, v in f['sin'].items():
        marca = '  <-- mayor dependencia' if a == f['peor_artista'] else ''
        print(f'   sin {a:30s} -> {v:6,d} personas  ({100*(v-f["publico"])/f["publico"]:+6.1f}%){marca}')
    print(f'   ratio de robustez = {f["ratio"]:.3f}')

print('\nRESUMEN')
print(f'{"Noche":10s} {"Base":>8s} {"Peor caso":>10s} {"Ratio":>7s}  Artista crítico')
for f in cancel:
    print(f'{f["noche"]:10s} {f["publico"]:8,d} {f["peor"]:10,d} {f["ratio"]:7.3f}  {f["peor_artista"]}')

peor_noche = min(cancel, key=lambda f: f['ratio'])
print(f'\nNoche más dependiente de un solo artista: {peor_noche["noche"]} '
      f'(cae {100*(1-peor_noche["ratio"]):.1f}% si cancela {peor_noche["peor_artista"]})')

---
# Pregunta 1(c) II — Condiciones de robustez

Se incorporan al modelo cuatro condiciones alternativas y se comparan en público y costo:

| # | Condición |
| :--- | :--- |
| A1 | Ningún artista aporta más del 50% del público de su noche ($\alpha=0{,}50$) |
| A2 | Ídem con exigencia 55% ($\alpha=0{,}55$) |
| A3 | Al menos un artista grande por noche |
| A4 | A lo más un artista pequeño por noche |

La condición de robustez se linealiza así ($\alpha$ es un parámetro y $\text{Pub}(c\setminus\{a\},n,r)$
también es un parámetro precalculado):

$$\text{Pub}(c\setminus\{a\},n,r)\ \ge\ \alpha\,\text{Pub}_{cnr}-12.000\,(1-X_{cnr})$$

In [ ]:
# =====================================================================
# (c) II  Alternativas de robustez
# =====================================================================
ALTERNATIVAS = [
    ('A1: robustez alpha = 0,50', dict(alpha=0.50)),
    ('A2: robustez alpha = 0,55', dict(alpha=0.55)),
    ('A3: al menos 1 artista grande por noche', dict(min_grandes=1)),
    ('A4: a lo más 1 artista pequeño por noche', dict(max_pequenos=1)),
]

alt = []
print(f'{"Alternativa":42s} {"Público":>8s} {"Δ base":>7s} {"Costo":>6s} {"Peor noche":>11s} {"Ratio":>6s}')
print(f'{"Base":42s} {base["pub_total"]:8,d} {0:7d} {base["costo_total"]:6d} '
      f'{min(f["peor"] for f in cancel):11,d} {min(f["ratio"] for f in cancel):6.3f}')
for nombre, kw in ALTERNATIVAS:
    s = resolver(PRESUPUESTO, **kw)
    if s is None:
        print(f'{nombre:42s} INFACTIBLE')
        continue
    det = analisis_cancelaciones(s)
    peor = min(f['peor'] for f in det)
    ratio = min(f['ratio'] for f in det)
    alt.append({'nombre': nombre, 'sol': s, 'detalle': det, 'peor': peor, 'ratio': ratio})
    print(f'{nombre:42s} {s["pub_total"]:8,d} {s["pub_total"]-base["pub_total"]:+7d} '
          f'{s["costo_total"]:6d} {peor:11,d} {ratio:6.3f}')
    assert not validar(s), validar(s)

print('\nParrillas resultantes:')
for a in alt:
    print(f'  {a["nombre"]:42s}', [a['sol']['pub_noche'][n] for n in NOCHES])

# --- gráfico comparativo ---
import matplotlib.pyplot as plt
nombres = ['Base'] + [a['nombre'].split(':')[0] for a in alt]
tot = [base['pub_total']] + [a['sol']['pub_total'] for a in alt]
pes = [min(f['peor'] for f in cancel)] + [a['peor'] for a in alt]
x = range(len(nombres)); w = 0.38
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar([i - w/2 for i in x], tot, w, label='Público total', color='#1565c0')
ax.bar([i + w/2 for i in x], pes, w, label='Peor noche tras una cancelación', color='#ef6c00')
for i, (t, pv) in enumerate(zip(tot, pes)):
    ax.text(i - w/2, t + 250, f'{t:,}', ha='center', fontsize=7)
    ax.text(i + w/2, pv + 250, f'{pv:,}', ha='center', fontsize=7)
ax.set_xticks(list(x)); ax.set_xticklabels(nombres); ax.set_ylim(0, 23000)
ax.set_ylabel('Personas'); ax.legend(frameon=False, fontsize=8); ax.grid(axis='y', alpha=.3)
ax.set_title('(c) II — Público total vs. robustez')
plt.tight_layout(); plt.show()

## (c) III — Recomendación

**Se recomienda incorporar A4 (a lo más un artista pequeño por noche).**

- **Se gana:** el peor escenario de cancelación sube de 1.544 a 2.599 personas (**+68%**) y el costo
  baja de 249 a 247 MM$, con una pérdida de público total de **1 persona**.
- **Se pierde:** flexibilidad para "rellenar" una noche con dos artistas pequeños, que en este caso no
  aportaba público.
- **Por qué no A1/A2:** cuestan 734–997 personas (hasta 5% del público) y dejan parrillas muy
  desbalanceadas (el viernes cae a ~3.980).
- **Por qué no A3:** es casi gratis (−5 personas) pero **no reduce la dependencia**: el ratio mínimo
  baja a 0,254, porque tener un artista grande no protege si ese artista concentra la convocatoria.

Si se quisiera una garantía formal de que ninguna noche cae bajo la mitad de su público, la opción es
A1 con $\alpha=0{,}50$, asumiendo un costo de ~735 personas.

---
# Pregunta 1(d) I — Sensibilidad al presupuesto

Se resuelve el modelo base (con desempate) para presupuestos de 180, 220, 260, 300 y 340 MM$
(se agrega 250 como referencia).

In [ ]:
# =====================================================================
# (d) I  Barrido de presupuesto
# =====================================================================
PRESUPUESTOS = [180, 220, 250, 260, 300, 340]
barrido = []
print(f'{"B":>5s} {"Público":>9s} {"Viernes":>9s} {"Sábado":>9s} {"Domingo":>9s} {"Seguridad":>10s} {"Costo":>6s}')
for B in PRESUPUESTOS:
    s = resolver(B)
    barrido.append(s)
    p = s['pub_noche']
    print(f'{B:5d} {s["pub_total"]:9,d} {p["Viernes"]:9,d} {p["Sábado"]:9,d} {p["Domingo"]:9,d} '
          f'{("SÍ" if s["seguridad"] else "no"):>10s} {s["costo_total"]:6d}')

print('\nParrilla por presupuesto:')
for s in barrido:
    print(f'  {s["presupuesto"]:3d} MM$:', [(n, r) for n, r, _ in s['parrilla']])

# --- gráfico ---
import numpy as np
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 3.8))
x = np.arange(len(PRESUPUESTOS)); bottom = np.zeros(len(PRESUPUESTOS))
for color, n in zip(['#1565c0', '#c62828', '#2e7d32'], NOCHES):
    vals = np.array([s['pub_noche'][n] for s in barrido])
    ax.bar(x, vals, 0.6, bottom=bottom, label=n, color=color, alpha=.9)
    bottom += vals
for xi, s, tot in zip(x, barrido, bottom):
    ax.text(xi, tot + 350, f'{tot:,.0f}', ha='center', fontsize=8, fontweight='bold')
ax.axhline(UMBRAL_SEGURIDAD, ls='--', lw=1, color='k', alpha=.6)
ax.text(len(x) - 0.45, UMBRAL_SEGURIDAD + 450, 'umbral seguridad', ha='right', fontsize=7)
ax.set_xticks(x)
ax.set_xticklabels([f'{s["presupuesto"]} MM$\n' + ('con seguridad' if s['seguridad'] else 'sin seguridad')
                    for s in barrido], fontsize=7.5)
ax.set_ylabel('Público esperado total'); ax.set_ylim(0, 27500)
ax.legend(title='Noche', frameon=False, loc='upper left')
ax.set_title('(d) I — Público total según presupuesto')
ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()

## (d) II — ¿Qué noche queda más desmejorada?

**El domingo.** Es la noche sistemáticamente más baja en el rango de presupuesto relevante (220–260 MM$)
y en la solución base aporta solo el 16,5% del público total.

Razones, desde la formulación:

1. **Factor de noche más bajo** ($\phi_D=0{,}85$): cada artista rinde 15% menos que el sábado. Como el
   presupuesto (R6) es único y compartido, el modelo compra público donde el rendimiento por peso es
   mayor y le deja al domingo el último peso disponible.
2. **Competencia por artistas escasos (R2):** los mejores artistas están disponibles las tres noches y
   solo pueden contratarse una vez; con $\phi_S=1{,}00$ el sábado los absorbe.
3. **Costo fijo por recinto:** crecer en el domingo exige saltar de Gimnasio (4.200, 8 MM$) a Medialuna
   (7.500, 18 MM$) o Plaza (12.000, 30 MM$), y ese salto compite con contratar un artista grande.
4. **El umbral de seguridad lo refuerza:** como pasar de 20.000 cuesta 25 MM$, el modelo maximiza justo
   bajo el umbral, y ese ajuste fino se hace sacrificando la noche de menor rendimiento.

## (d) III — Mejorar la noche más débil sin subir de 250 MM$

Se incorpora un **piso de público mínimo por noche**:

$$\sum_{(c,n,r):\,n=\text{domingo}}\text{Pub}_{cnr}\,X_{cnr}\ \ge\ m \qquad\text{(y análogamente para cada noche)}$$

In [ ]:
# =====================================================================
# (d) III  Piso de público por noche
# =====================================================================
CASOS_D3 = [
    ('Piso 5.000 por noche', dict(minimo_noche={n: 5000 for n in NOCHES})),
    ('Piso 6.000 por noche', dict(minimo_noche={n: 6000 for n in NOCHES})),
    ('Piso 6.500 por noche', dict(minimo_noche={n: 6500 for n in NOCHES})),
    ('Al menos 1 grande por noche', dict(min_grandes=1)),
]

print(f'{"Condición":32s} {"Viernes":>8s} {"Sábado":>8s} {"Domingo":>8s} {"Total":>8s} {"Δ":>6s} {"Costo":>6s}')
b = base
print(f'{"Base":32s} {b["pub_noche"]["Viernes"]:8,d} {b["pub_noche"]["Sábado"]:8,d} '
      f'{b["pub_noche"]["Domingo"]:8,d} {b["pub_total"]:8,d} {0:6d} {b["costo_total"]:6d}')
d3 = []
for nombre, kw in CASOS_D3:
    s = resolver(PRESUPUESTO, **kw)
    d3.append((nombre, s))
    p = s['pub_noche']
    print(f'{nombre:32s} {p["Viernes"]:8,d} {p["Sábado"]:8,d} {p["Domingo"]:8,d} '
          f'{s["pub_total"]:8,d} {s["pub_total"]-base["pub_total"]:+6d} {s["costo_total"]:6d}')

print('\nMejor compromiso: piso de 6.000 por noche -> el domingo pasa de '
      f'{base["pub_noche"]["Domingo"]:,} a {d3[1][1]["pub_noche"]["Domingo"]:,} personas '
      f'(+{100*(d3[1][1]["pub_noche"]["Domingo"]/base["pub_noche"]["Domingo"]-1):.0f}%) '
      f'perdiendo solo {base["pub_total"]-d3[1][1]["pub_total"]} personas en total.')

## (d) IV — Sensibilidad al nivel de exigencia

Para cada nivel $m$ se mide (i) el público total con 250 MM$ y (ii) el **presupuesto mínimo** necesario
para no perder el público total base (19.999). Esta última búsqueda se hace por **bisección**: el público
máximo es no decreciente en el presupuesto (aflojar R6 nunca puede empeorar el óptimo), de modo que se
obtiene el valor exacto con unas pocas resoluciones por nivel.

In [ ]:
# =====================================================================
# (d) IV  Sensibilidad al nivel de exigencia
# =====================================================================
NIVELES = [4000, 4500, 5000, 5500, 6000, 6200, 6400, 6600, 6800, 7000]
sens = []


def presupuesto_minimo(piso, objetivo, tope=420):
    """Menor presupuesto que permite alcanzar `objetivo` de público con el piso dado.

    El público máximo es no decreciente en el presupuesto (aflojar R6 nunca puede empeorar
    el óptimo), así que la búsqueda se hace por bisección en vez de barrer todo el rango.
    """
    def alcanza(B):
        s = resolver(B, minimo_noche=piso)
        return s is not None and s['pub_total'] >= objetivo

    if not alcanza(tope):
        return None
    lo, hi = PRESUPUESTO, tope
    while lo < hi:
        mid = (lo + hi) // 2
        if alcanza(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


print(f'{"Nivel":>6s} {"Público 250":>12s} {"Δ base":>8s} {"Domingo":>9s} {"B mínimo":>9s} {"Δ B":>6s}')
for m in NIVELES:
    piso = {n: m for n in NOCHES}
    s = resolver(PRESUPUESTO, minimo_noche=piso)
    Bmin = presupuesto_minimo(piso, base['pub_total'])
    sens.append({'nivel': m, 'sol': s, 'Bmin': Bmin})
    bmin_txt = f'{Bmin:d}' if Bmin is not None else '>420'
    db_txt = f'{Bmin - PRESUPUESTO:+d}' if Bmin is not None else '-'
    if s is None:
        print(f'{m:6,d} {"infactible":>12s} {"-":>8s} {"-":>9s} {bmin_txt:>9s} {db_txt:>6s}')
    else:
        print(f'{m:6,d} {s["pub_total"]:12,d} {s["pub_total"]-base["pub_total"]:+8d} '
              f'{s["pub_noche"]["Domingo"]:9,d} {bmin_txt:>9s} {db_txt:>6s}')

# --- gráfico ---
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 3.8))
niv = [d['nivel'] for d in sens]
tot = [d['sol']['pub_total'] if d['sol'] else float('nan') for d in sens]
dom = [d['sol']['pub_noche']['Domingo'] if d['sol'] else float('nan') for d in sens]
ax.plot(niv, tot, 'o-', color='#1565c0', label='Público total')
ax.plot(niv, dom, 's--', color='#2e7d32', label='Público del domingo')
ax.axhline(base['pub_total'], color='k', ls=':', lw=1, alpha=.6)
ax.text(4020, base['pub_total'] + 150, f'base {base["pub_total"]:,}', fontsize=7)
primera_inf = next((i for i, d in enumerate(sens) if d['sol'] is None), None)
if primera_inf is not None:
    ax.axvspan(niv[primera_inf] - 100, niv[-1] + 100, color='#e0e0e0', alpha=.6)
    ax.text(niv[primera_inf] + 100, 11000, 'infactible con 250 MM$', fontsize=7, color='#555')
for xi, t in zip(niv, tot):
    if t == t:
        ax.text(xi, t + 250, f'{t:,.0f}', ha='center', fontsize=6.5)
ax.set_xlabel('Público mínimo exigido por noche (personas)')
ax.set_ylabel('Personas'); ax.set_ylim(0, 22500)
ax.legend(frameon=False, fontsize=8)
ax.set_title('(d) IV — Sensibilidad al nivel de exigencia (250 MM$)')
ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Conclusión de (d) IV

**Se recomienda fijar el piso en 6.000 personas por noche.**

- Cuesta solo **12 personas** (−0,06%) de público total.
- Sube el domingo de 3.297 a **6.219** (+89%), eliminando la noche desmejorada.
- Con **+4 MM$** de presupuesto (254 en total) se recupera íntegramente el público total base, llegando
  a 20.000 personas.
- Sobre 6.400 el costo se dispara (−225 personas) y desde 6.600 la exigencia es **infactible con
  250 MM$** (requiere al menos 278 MM$); esos niveles no se justifican.

---

### Resumen final

| Pregunta | Resultado |
| :--- | :--- |
| (b) Público total | **19.999 personas** |
| (b) Costo total | **249 MM$** (193 artistas + 56 recintos) |
| (b) Plan de seguridad | **No se requiere** |
| (c) Noche más frágil | **Sábado** (cae 71,8% sin Camila Arrebol) |
| (c) Condición recomendada | **A4: a lo más 1 pequeño por noche** (−1 persona, peor caso +68%) |
| (d) Noche más desmejorada | **Domingo** |
| (d) Condición recomendada | **Piso de 6.000 por noche** (−12 personas, domingo +89%) |